NAV cleaning
Transaction cleaning
Performance cleaning
Data quality checks
Validation outputs

In [2]:
from pathlib import Path
import pandas as pd

RAW_PATH = Path("../data/raw")

In [3]:
for file in RAW_PATH.glob("*.csv"):
    
    df = pd.read_csv(file)

    print("\n" + "="*70)
    print(file.name)

    print("\nShape")
    print(df.shape)

    print("\nColumns")
    print(df.columns.tolist())

    print("\nMissing Values")
    print(df.isnull().sum())

    print("\nDuplicates")
    print(df.duplicated().sum())

    print("\nData Types")
    print(df.dtypes)


01_fund_master.csv

Shape
(40, 15)

Columns
['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']

Missing Values
amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

Duplicates
0

Data Types
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64


In [4]:
nav = pd.read_csv(
    "../data/raw/02_nav_history.csv"
)

In [5]:
nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [6]:
nav.columns

Index(['amfi_code', 'date', 'nav'], dtype='object')

In [7]:
nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce"
)

In [8]:
nav["date"].isna().sum()

np.int64(0)

In [9]:
nav = nav.dropna(
    subset=["date"]
)

In [10]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

In [11]:
nav = nav.drop_duplicates(
    subset=["amfi_code","date"]
)

In [12]:
nav = nav[
    nav["nav"] > 0
]

In [13]:
cleaned_groups = []

for code, grp in nav.groupby("amfi_code"):

    grp = grp.set_index("date")

    full_dates = pd.date_range(
        start=grp.index.min(),
        end=grp.index.max()
    )

    grp = grp.reindex(full_dates)

    grp["nav"] = grp["nav"].ffill()

    grp["amfi_code"] = code

    cleaned_groups.append(grp)

nav = pd.concat(cleaned_groups)

In [14]:
nav.to_csv(
    "../data/processed/clean_nav_history.csv",
    index=False
)


In [15]:
txn = pd.read_csv(
    "../data/raw/08_investor_transactions.csv"
)

In [16]:
txn["transaction_type"].unique()

array(['SIP', 'Redemption', 'Lumpsum'], dtype=object)

In [17]:
txn["transaction_type"] = (
    txn["transaction_type"]
    .str.strip()
    .str.lower()
)

In [18]:
mapping = {
    "sip":"SIP",
    "lumpsum":"Lumpsum",
    "redemption":"Redemption"
}

txn["transaction_type"] = txn[
    "transaction_type"
].map(mapping)

In [23]:
print(txn.columns)

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='object')


In [24]:
txn["transaction_type"] = (
    txn["transaction_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [25]:
mapping = {
    "sip": "SIP",
    "lumpsum": "Lumpsum",
    "redemption": "Redemption"
}

txn["transaction_type"] = txn["transaction_type"].replace(mapping)

In [26]:
txn = txn[txn["amount_inr"] > 0]

In [27]:
txn["amount_inr"].describe()

count     32778.000000
mean     107437.318628
std      150415.905084
min         400.000000
25%        3153.000000
50%       17782.500000
75%      189324.250000
max      597498.000000
Name: amount_inr, dtype: float64

In [28]:
txn["kyc_status"].value_counts()

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [29]:
valid = ["Verified", "Pending", "Rejected"]

invalid_kyc = txn[
    ~txn["kyc_status"].isin(valid)
]

print(invalid_kyc.shape)

(0, 13)


In [30]:
txn["state"] = (
    txn["state"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [31]:
txn["city"] = (
    txn["city"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [32]:
txn.to_csv(
    "../data/processed/clean_transactions.csv",
    index=False
)

In [33]:
perf = pd.read_csv("../data/raw/07_scheme_performance.csv")
print(perf.columns.tolist())

['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [34]:
perf = pd.read_csv(
    "../data/raw/07_scheme_performance.csv"
)

In [35]:
perf.isnull().sum()

amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

In [36]:
perf = perf.drop_duplicates()

In [37]:
numeric_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct",
    "morningstar_rating"
]

In [38]:
for col in numeric_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors="coerce"
    )

In [39]:
perf[numeric_cols].dtypes

return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
dtype: object

In [40]:
perf[
    perf["return_1yr_pct"].isna()
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [41]:
perf[
    perf["return_3yr_pct"].isna()
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [42]:
perf[
    perf["return_5yr_pct"].isna()
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [43]:
anomalies = perf[
    (perf["return_1yr_pct"] > 100)
    |
    (perf["return_1yr_pct"] < -100)
]

In [44]:
anomalies.to_csv(
    "../reports/performance_anomalies.csv",
    index=False
)

In [45]:
bad_expense = perf[
    (perf["expense_ratio_pct"] < 0.1)
    |
    (perf["expense_ratio_pct"] > 2.5)
]

In [46]:
bad_expense[
    [
        "scheme_name",
        "expense_ratio_pct"
    ]
]

,scheme_name,expense_ratio_pct


In [47]:
bad_expense.to_csv(
    "../reports/bad_expense_ratio.csv",
    index=False
)

In [48]:
perf["negative_sharpe"] = (
    perf["sharpe_ratio"] < 0
)

In [49]:
perf["negative_sharpe"].value_counts()

negative_sharpe
False    40
Name: count, dtype: int64

In [50]:
perf = perf[
    perf["aum_crore"] >= 0
]

In [51]:

perf["morningstar_rating"].value_counts()

morningstar_rating
5    17
4    16
3     7
Name: count, dtype: int64

In [52]:
perf[
    ~perf["morningstar_rating"].isin(
        [1,2,3,4,5]
    )
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,negative_sharpe


In [53]:
perf["risk_grade"].value_counts()

risk_grade
Moderate           16
High                8
Very High           6
Low                 6
Moderately High     4
Name: count, dtype: int64

In [54]:
perf["risk_grade"] = (
    perf["risk_grade"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [55]:
perf.to_csv(
    "../data/processed/clean_performance.csv",
    index=False
)

In [56]:
import sqlite3

conn = sqlite3.connect(
    "../data/db/bluestock_mf.db"
)

cursor = conn.cursor()

cursor.execute(
    "SELECT COUNT(*) FROM fact_nav"
)

print(cursor.fetchone())

(64320,)


In [57]:
cursor.execute(
    "SELECT COUNT(*) FROM fact_transactions"
)

print(cursor.fetchone())

(32778,)


In [58]:
cursor.execute(
    "SELECT COUNT(*) FROM fact_performance"
)

print(cursor.fetchone())

(40,)
